In [1]:
from astropy.io import fits
import numpy as np
import os


def read_in_weights(infile: str):
    """Reads FITS data and header"""
    with fits.open(infile) as hdu:
        data = hdu[0].data
        header = hdu[0].header
    return data, header


def mask_data(data: np.ndarray, limit: float) -> np.ndarray:
    """Creates a binary mask: 1 where data >= limit, else 0"""
    mask = np.zeros_like(data, dtype=np.uint8)
    mask[data >= limit] = 1
    return mask


def save_mask(mask: np.ndarray, header, outfile: str):
    """Saves the mask as a FITS file"""
    hdu = fits.PrimaryHDU(data=mask, header=header)
    hdu.writeto(outfile, overwrite=True)
    print(f"Saved masked image: {outfile}")


def create_and_save_masks(weights: list[tuple], outdir: str):
    """Creates and saves individual masked FITS images"""
    
    os.makedirs(outdir, exist_ok=True)

    for weight_file, limit in weights:
        print(f"\nProcessing: {weight_file}")
        print(f"Threshold: {limit}")

        data, header = read_in_weights(weight_file)
        mask = mask_data(data, limit)

        print(f"Pixels above threshold: {np.sum(mask)}")

        # Create output filename
        base = os.path.basename(weight_file)
        name = base.replace("weight", "weighted_mask")
        outfile = os.path.join(outdir, name)

        save_mask(mask, header, outfile)


if __name__ == '__main__':
    WEIGHTS = [
        ('/Users/aishwarya/Desktop/Lyman_alpha_2/Mosaiced_images/Trim2deg/trim2deg_weight_i.fits', 0.0008),
        ('/Users/aishwarya/Desktop/Lyman_alpha_2/Mosaiced_images/Trim2deg/trim2deg_weight_z.fits', 0.001),
        ('/Users/aishwarya/Desktop/Lyman_alpha_2/Y_band/Trim/trim2deg_weight_y.fits', 1.97e-3)
    ]

    OUTPUT_DIR = '/Users/aishwarya/Desktop/Mask'

    create_and_save_masks(WEIGHTS, OUTPUT_DIR)


Processing: /Users/aishwarya/Desktop/Lyman_alpha_2/Mosaiced_images/Trim2deg/trim2deg_weight_i.fits
Threshold: 0.0008
Pixels above threshold: 556075003
Saved masked image: /Users/aishwarya/Desktop/Mask/trim2deg_weighted_mask_i.fits

Processing: /Users/aishwarya/Desktop/Lyman_alpha_2/Mosaiced_images/Trim2deg/trim2deg_weight_z.fits
Threshold: 0.001
Pixels above threshold: 548643227
Saved masked image: /Users/aishwarya/Desktop/Mask/trim2deg_weighted_mask_z.fits

Processing: /Users/aishwarya/Desktop/Lyman_alpha_2/Y_band/Trim/trim2deg_weight_y.fits
Threshold: 0.00197
Pixels above threshold: 565639157
Saved masked image: /Users/aishwarya/Desktop/Mask/trim2deg_weighted_mask_y.fits


# Mask CDFS

In [2]:
from astropy.io import fits
import numpy as np
import os


def create_nan_mask(infile: str):
    """Convert NaNs → 0 and valid pixels → 1, save as FITS mask"""

    # Read FITS
    with fits.open(infile) as hdul:
        data = hdul[0].data
        header = hdul[0].header

    # Create mask: 1 = good, 0 = NaN
    mask = np.ones_like(data, dtype=np.uint8)
    mask[np.isnan(data)] = 0

    # Output path (same directory)
    outdir = "/Users/aishwarya/Desktop/Mask"
    outfile = os.path.join(outdir, "cdfs_mask.fits")

    # Save mask
    hdu = fits.PrimaryHDU(data=mask, header=header)
    hdu.writeto(outfile, overwrite=True)

    print(f"Mask saved to: {outfile}")
    print(f"Good pixels: {np.sum(mask)}")
    print(f"Bad pixels (NaN): {np.sum(mask == 0)}")


if __name__ == "__main__":
    infile = "/Users/aishwarya/Desktop/new_cdfs/trimmed_2deg/trim2deg_y_sci.fits"
    create_nan_mask(infile)

Mask saved to: /Users/aishwarya/Desktop/Mask/cdfs_mask.fits
Good pixels: 376184737
Bad pixels (NaN): 335104163
